In [5]:
# Install: pip install datasets openai pandas pillow requests

from datasets import load_dataset
import pandas as pd
from pathlib import Path
import base64
from openai import OpenAI
import requests
from PIL import Image
import time
import json
import os
from io import BytesIO

# --- 1. SETUP ---
# Initialize OpenAI client (Ensure OPENAI_API_KEY is in your environment variables)
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set in your environment variables.")

# --- 2. DATA LOADING ---
print("Loading product dataset...")
try:
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:10]")
    products_df = pd.DataFrame(dataset)
    print(f"✓ Loaded {len(products_df)} products")
    print("Dataset columns:", products_df.columns.tolist())

except Exception as e:
    print(f"⚠ Could not load dataset: {e}")
    print("Using fallback local data instead...")
    products_df = pd.DataFrame([
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        }
    ])

# Normalize product name column so code works for both dataset and fallback
if 'productDisplayName' not in products_df.columns and 'name' in products_df.columns:
    products_df['productDisplayName'] = products_df['name']

# --- 3. VISION INTEGRATION FUNCTIONS ---
def encode_pil_image(pil_image):
    """Encodes a PIL image to base64 for the API."""
    buffer = BytesIO()
    pil_image.convert("RGB").save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

def encode_image_file(image_path):
    """Encodes a local image file to base64 for the API."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def get_base64_image(product_row):
    """
    Returns base64 image data from either:
    - Hugging Face dataset image column
    - local image_path fallback
    """
    try:
        # Case 1: Hugging Face dataset image column exists
        if 'image' in product_row and product_row['image'] is not None:
            img = product_row['image']

            if isinstance(img, Image.Image):
                return encode_pil_image(img)

            elif isinstance(img, dict):
                if 'bytes' in img and img['bytes'] is not None:
                    pil_img = Image.open(BytesIO(img['bytes']))
                    return encode_pil_image(pil_img)

                elif 'path' in img and img['path'] is not None:
                    return encode_image_file(img['path'])

        # Case 2: local file path fallback
        if 'image_path' in product_row and pd.notna(product_row['image_path']):
            return encode_image_file(product_row['image_path'])

        raise ValueError("No valid image source found.")

    except Exception as e:
        raise ValueError(f"Image encoding failed: {e}")

def generate_product_description(product_row):
    """Uses OpenAI vision model to generate a compelling product description."""
    try:
        base64_image = get_base64_image(product_row)

        product_name = product_row.get('productDisplayName', product_row.get('name', 'this item'))
        price = product_row.get('price', 'N/A')
        category = product_row.get('masterCategory', product_row.get('category', 'General'))

        prompt = f"""
Write a compelling, SEO-friendly product listing for this product.

Product name: {product_name}
Category: {category}
Price: {price}

Instructions:
- Write 80-120 words
- Focus on visible features and likely benefits
- Keep it professional and suitable for an online store
- Do not invent brand names or technical details not visible in the image
"""

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "system",
                    "content": "You are a professional e-commerce copywriter."
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=300
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"Error generating description: {e}"

# --- 4. EXECUTION ---
print("Generating descriptions...")

sample_df = products_df.head(5).copy()
sample_df['description'] = sample_df.apply(generate_product_description, axis=1)

print("\n✓ Processing Complete!")

name_col = 'productDisplayName' if 'productDisplayName' in sample_df.columns else 'name'
print(sample_df[[name_col, 'description']].head())

Loading product dataset...
✓ Loaded 10 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']
Generating descriptions...

✓ Processing Complete!
                              productDisplayName  \
0               Turtle Check Men Navy Blue Shirt   
1             Peter England Men Party Blue Jeans   
2                       Titan Women Silver Watch   
3  Manchester United Men Solid Black Track Pants   
4                          Puma Men Grey T-shirt   

                                         description  
0  Elevate your wardrobe with the Turtle Check Me...  
1  Elevate your party style with Peter England Me...  
2  Enhance your elegance with the Titan Women Sil...  
3  Elevate your sportswear collection with the Ma...  
4  **Puma Men Grey T-shirt**\n\nElevate your casu...  
